In [0]:
# ============================================================
# Notebook  : transform_flight_prices
# Purpose   : Read Bronze table, apply DQ checks,
#             deduplicate and MERGE into Silver table
# Author    : FlightPriceTracker
# Version   : v1.0
# ============================================================

import uuid
import traceback
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    trim, upper, row_number
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, TimestampType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ============================================================
# SECTION 1 — INIT
# ============================================================

spark = SparkSession.builder \
    .appName("FlightPriceTracker_Transform") \
    .getOrCreate()

# ============================================================
# SECTION 2 — CONSTANTS
# ============================================================

RUN_ID = spark.sql("SELECT run_id FROM workspace.flight_tracker.flight_prices_raw ORDER BY ingested_at DESC LIMIT 1").collect()[0][0]
JOB_NAME         = "FlightPriceTracker"
NOTEBOOK_NAME    = "transform_flight_prices"
TASK_SEQUENCE    = 2
TRIGGERED_BY     = "SCHEDULER"
DQ_CHECK_VERSION = "v1"
DATABASE         = "workspace.flight_tracker"
RAW_TABLE        = f"{DATABASE}.flight_prices_raw"
CLEAN_TABLE      = f"{DATABASE}.flight_prices_clean"
AUDIT_TABLE      = f"{DATABASE}.pipeline_audit_log"

def now():
    return datetime.now(timezone.utc)

print(f"RUN_ID     : {RUN_ID}")
print(f"NOTEBOOK   : {NOTEBOOK_NAME}")
print(f"START TIME : {now()}")

# ============================================================
# SECTION 3 — AUDIT SCHEMA
# ============================================================

audit_schema = StructType([
    StructField("run_id",               StringType(),    True),
    StructField("job_name",             StringType(),    True),
    StructField("notebook_name",        StringType(),    True),
    StructField("task_sequence",        IntegerType(),   True),
    StructField("cluster_id",           StringType(),    True),
    StructField("start_time",           TimestampType(), True),
    StructField("end_time",             TimestampType(), True),
    StructField("duration_seconds",     IntegerType(),   True),
    StructField("status",               StringType(),    True),
    StructField("records_read",         IntegerType(),   True),
    StructField("records_written",      IntegerType(),   True),
    StructField("records_skipped",      IntegerType(),   True),
    StructField("records_failed",       IntegerType(),   True),
    StructField("error_message",        StringType(),    True),
    StructField("error_stacktrace",     StringType(),    True),
    StructField("retry_attempt",        IntegerType(),   True),
    StructField("triggered_by",         StringType(),    True),
    StructField("databricks_job_run_id",StringType(),    True),
    StructField("created_at",           TimestampType(), True),
])

# ============================================================
# SECTION 4 — AUDIT HELPER
# ============================================================

def log_audit(
    run_id, job_name, notebook_name, task_sequence,
    start_time, end_time, status,
    records_read=0, records_written=0,
    records_skipped=0, records_failed=0,
    error_message=None, error_stacktrace=None,
    retry_attempt=0, triggered_by="SCHEDULER"
):
    duration = int((end_time - start_time).total_seconds())
    audit_data = [{
        "run_id"               : run_id,
        "job_name"             : job_name,
        "notebook_name"        : notebook_name,
        "task_sequence"        : task_sequence,
        "cluster_id"           : "serverless",
        "start_time"           : start_time,
        "end_time"             : end_time,
        "duration_seconds"     : duration,
        "status"               : status,
        "records_read"         : records_read,
        "records_written"      : records_written,
        "records_skipped"      : records_skipped,
        "records_failed"       : records_failed,
        "error_message"        : error_message if error_message else "",
        "error_stacktrace"     : error_stacktrace if error_stacktrace else "",
        "retry_attempt"        : retry_attempt,
        "triggered_by"         : triggered_by,
        "databricks_job_run_id": "",
        "created_at"           : now()
    }]
    audit_df = spark.createDataFrame(audit_data, schema=audit_schema)
    audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)
    print(f"[AUDIT] {status} logged for {notebook_name}")

# ============================================================
# SECTION 5 — CLEANING
# ============================================================

def apply_cleaning(df):
    return df \
        .withColumn("route_id",             trim(upper(col("route_id")))) \
        .withColumn("origin",               trim(upper(col("origin")))) \
        .withColumn("destination",          trim(upper(col("destination")))) \
        .withColumn("airline",              trim(col("airline"))) \
        .withColumn("airline_code",         trim(upper(col("airline_code")))) \
        .withColumn("flight_number",        trim(upper(col("flight_number")))) \
        .withColumn("cabin_class",          trim(upper(col("cabin_class")))) \
        .withColumn("price_usd",            col("price_usd").cast("double")) \
        .withColumn("num_stops",            col("num_stops").cast("int")) \
        .withColumn("duration_minutes",     col("duration_minutes").cast("int")) \
        .withColumn("seats_available",      col("seats_available").cast("int")) \
        .withColumn("baggage_allowance_kg", col("baggage_allowance_kg").cast("int")) \
        .withColumn("is_refundable",        col("is_refundable").cast("boolean")) \
        .withColumn("dq_check_version",     lit(DQ_CHECK_VERSION)) \
        .withColumn("transformed_at",       current_timestamp())

# ============================================================
# SECTION 6 — DQ CHECKS
# ============================================================

def apply_dq_checks(df):
    df = df.withColumn(
        "invalid_reason",
        when(col("route_id").isNull() | (trim(col("route_id")) == ""),
             "route_id is null or empty")
        .when(col("airline").isNull() | (trim(col("airline")) == ""),
              "airline is null or empty")
        .when(col("departure_datetime").isNull(),
              "departure_datetime is null")
        .when(col("arrival_datetime").isNull(),
              "arrival_datetime is null")
        .when(col("arrival_datetime") <= col("departure_datetime"),
              "arrival_datetime is before or equal to departure_datetime")
        .when(col("flight_number").isNull() | (trim(col("flight_number")) == ""),
              "flight_number is null or empty")
        .when(col("num_stops") < 0,
              "num_stops is negative")
        .otherwise(None)
    )

    df = df.withColumn(
        "is_valid",
        when(col("invalid_reason").isNull(), lit(True)).otherwise(lit(False))
    )

    valid_count   = df.filter(col("is_valid") == True).count()
    invalid_count = df.filter(col("is_valid") == False).count()
    print(f"[DQ] Valid: {valid_count} | Invalid: {invalid_count}")

    return df

# ============================================================
# SECTION 7 — DEDUPLICATION
# ============================================================

def apply_deduplication(df):
    window_spec = Window.partitionBy("record_checksum").orderBy(col("ingested_at").desc())

    df = df.withColumn("row_num", row_number().over(window_spec))
    df = df.withColumn(
        "is_duplicate",
        when(col("row_num") > 1, lit(True)).otherwise(lit(False))
    ).drop("row_num")

    dup_count    = df.filter(col("is_duplicate") == True).count()
    unique_count = df.filter(col("is_duplicate") == False).count()
    print(f"[DEDUP] Unique: {unique_count} | Duplicates: {dup_count}")

    return df

# ============================================================
# SECTION 8 — MERGE INTO SILVER
# ============================================================

def merge_into_silver(df):
    silver_table = DeltaTable.forName(spark, CLEAN_TABLE)

    silver_table.alias("target").merge(
        df.alias("source"),
        "target.record_checksum = source.record_checksum"
    ).whenNotMatchedInsertAll() \
     .execute()

    print(f"[MERGE] Merge into {CLEAN_TABLE} complete")

# ============================================================
# SECTION 9 — MAIN EXECUTION
# ============================================================

start_time      = now()
records_read    = 0
records_written = 0
records_skipped = 0
records_failed  = 0
pipeline_status = "SUCCESS"
pipeline_error  = None
pipeline_trace  = None

try:
    print(f"\n[READ] Reading Bronze records for run_id: {RUN_ID}")
    raw_df       = spark.sql(f"SELECT * FROM {RAW_TABLE} WHERE run_id = '{RUN_ID}'")
    records_read = raw_df.count()
    print(f"[READ] Records found: {records_read}")

    if records_read == 0:
        print("[READ] No records found for this run.")
        pipeline_status = "PARTIAL"

    else:
        cleaned_df = apply_cleaning(raw_df)
        dq_df      = apply_dq_checks(cleaned_df)
        final_df   = apply_deduplication(dq_df)

        records_written = final_df.filter(
            (col("is_valid") == True) & (col("is_duplicate") == False)
        ).count()

        records_skipped = final_df.filter(
            (col("is_valid") == False) | (col("is_duplicate") == True)
        ).count()

        merge_into_silver(final_df)

        print(f"[SUMMARY] Read: {records_read} | Written: {records_written} | Skipped: {records_skipped}")

except Exception as e:
    pipeline_status = "FAILED"
    pipeline_error  = str(e)
    pipeline_trace  = traceback.format_exc()
    print(f"[ERROR] {pipeline_error}")

finally:
    end_time = now()
    log_audit(
        run_id           = RUN_ID,
        job_name         = JOB_NAME,
        notebook_name    = NOTEBOOK_NAME,
        task_sequence    = TASK_SEQUENCE,
        start_time       = start_time,
        end_time         = end_time,
        status           = pipeline_status,
        records_read     = records_read,
        records_written  = records_written,
        records_skipped  = records_skipped,
        records_failed   = records_failed,
        error_message    = pipeline_error,
        error_stacktrace = pipeline_trace,
        triggered_by     = TRIGGERED_BY
    )
    print(f"\n[DONE] {NOTEBOOK_NAME} — Status: {pipeline_status}")
    print(f"[DONE] Read: {records_read} | Written: {records_written} | Skipped: {records_skipped} | Failed: {records_failed}")
    dbutils.notebook.exit(RUN_ID)

In [0]:
%sql
SELECT *
FROM workspace.flight_tracker.flight_prices_clean;

run_id,route_id,origin,destination,airline,airline_code,aircraft_type,flight_number,num_stops,stop_locations,departure_datetime,arrival_datetime,duration_minutes,price_usd,cabin_class,seats_available,baggage_allowance_kg,is_refundable,ingested_at,record_checksum,is_valid,invalid_reason,is_duplicate,dq_check_version,transformed_at
7b962786-7249-4d8c-8b6b-7813f209c463,DEL-BOM,DEL,BOM,Air India,AI,Airbus A320 NEO,AI 2433,0,,2026-06-29T16:00:00.000Z,2026-06-29T18:25:00.000Z,145,0.0,ECONOMY,0,0,false,2026-06-29T11:42:04.023Z,050f2af96614ee03a03fb1aab098f86b,true,null,false,v1,2026-06-29T11:51:57.247Z
7b962786-7249-4d8c-8b6b-7813f209c463,DEL-BOM,DEL,BOM,SpiceJet,SG,Boeing 737-700,SG 800,0,,2026-06-29T15:30:00.000Z,2026-06-29T17:40:00.000Z,130,0.0,ECONOMY,0,0,false,2026-06-29T11:42:04.023Z,080fa621b5bf680de25c2ca2fb8a222e,true,null,false,v1,2026-06-29T11:51:57.247Z
7b962786-7249-4d8c-8b6b-7813f209c463,DEL-BOM,DEL,BOM,IndiGo,6E,Airbus A320,6E 853,0,,2026-06-29T15:15:00.000Z,2026-06-29T17:30:00.000Z,135,0.0,ECONOMY,0,0,false,2026-06-29T11:42:04.023Z,0f99b295cf39c546839d663cfac93c91,true,null,false,v1,2026-06-29T11:51:57.247Z
7b962786-7249-4d8c-8b6b-7813f209c463,DEL-BOM,DEL,BOM,Air India,AI,Airbus A320 NEO,AI 2977,0,,2026-06-29T13:30:00.000Z,2026-06-29T15:55:00.000Z,145,0.0,ECONOMY,0,0,false,2026-06-29T11:42:04.023Z,28831d8610a8ee43ea2f53a61e9fbbca,true,null,false,v1,2026-06-29T11:51:57.247Z
7b962786-7249-4d8c-8b6b-7813f209c463,DEL-BOM,DEL,BOM,IndiGo,6E,Airbus A320,6E 6114,0,,2026-06-29T17:15:00.000Z,2026-06-29T19:35:00.000Z,140,0.0,ECONOMY,0,0,false,2026-06-29T11:42:04.023Z,2cd0919c6d9260762e2df4bf3120a6e5,true,null,false,v1,2026-06-29T11:51:57.247Z
7b962786-7249-4d8c-8b6b-7813f209c463,DEL-BOM,DEL,BOM,Starlight Airline,QP,Boeing 737,QP 1826,0,,2026-06-29T14:25:00.000Z,2026-06-29T16:50:00.000Z,145,0.0,ECONOMY,0,0,false,2026-06-29T11:42:04.023Z,2f81ec7d0fb6f1128cb4e8442dec57df,true,null,false,v1,2026-06-29T11:51:57.247Z
7b962786-7249-4d8c-8b6b-7813f209c463,DEL-BOM,DEL,BOM,Air India,AI,Airbus A320 NEO,AI 2957,0,,2026-06-29T15:00:00.000Z,2026-06-29T17:35:00.000Z,155,0.0,ECONOMY,0,0,false,2026-06-29T11:42:04.023Z,45bcc23d67c32b4092a0bfc12f27f114,true,null,false,v1,2026-06-29T11:51:57.247Z
7b962786-7249-4d8c-8b6b-7813f209c463,DEL-BOM,DEL,BOM,Air India,AI,Airbus A320 NEO,AI 2955,0,,2026-06-29T12:30:00.000Z,2026-06-29T14:55:00.000Z,145,0.0,ECONOMY,0,0,false,2026-06-29T11:42:04.022Z,6569a6b34cdcdbf2ed6c8ae5efac7acc,true,null,false,v1,2026-06-29T11:51:57.247Z
7b962786-7249-4d8c-8b6b-7813f209c463,DEL-BOM,DEL,BOM,IndiGo,6E,Airbus A320,6E 395,0,,2026-06-29T16:15:00.000Z,2026-06-28T18:35:00.000Z,-1300,0.0,ECONOMY,0,0,false,2026-06-29T11:42:04.023Z,6e6c22cd6ac0a13aadd2202a8e56b11a,false,arrival_datetime is before or equal to departure_datetime,false,v1,2026-06-29T11:51:57.247Z
7b962786-7249-4d8c-8b6b-7813f209c463,DEL-BOM,DEL,BOM,Starlight Airline,QP,Boeing 737,QP 1820,0,,2026-06-29T12:10:00.000Z,2026-06-29T14:20:00.000Z,130,0.0,ECONOMY,0,0,false,2026-06-29T11:42:04.022Z,6f1e2b8ce9e4316e62d6a141e41be4da,true,null,false,v1,2026-06-29T11:51:57.247Z
